In [10]:
:set -XArrows
:set -XTupleSections
:set -XFlexibleInstances
:set -XFlexibleContexts
:set -XInstanceSigs
import Control.Arrow
import Control.Category
import Prelude hiding (id, (.))
putStrLn "Setup done."

Setup done.

# Стрелки в Haskell

**Стрелки** (Arrows) — обобщение функций, монад и профункторов в едином интерфейсе.
Стрелка `arr a b` — это вычисление, преобразующее `a` в `b`, но мощнее
простой функции: может иметь состояние, ветвление, обратную связь.

> Hughes (2000): *"Generalising Monads to Arrows"* — стрелки позволяют абстрагировать
> больше вычислительных паттернов, чем монады.

> **📦 Зависимости**
> **Пакеты:** `base`
> **Расширения:** `Arrows` — proc-нотация для стрелок ([→](Extensions.ipynb#arrows)) · `FlexibleContexts` — произвольные ограничения в контекстах ([→](Extensions.ipynb#flexiblecontexts)) · `FlexibleInstances` — инстансы для конкретных типов-применений ([→](Extensions.ipynb#flexibleinstances)) · `InstanceSigs` — сигнатуры методов прямо в instance ([→](Extensions.ipynb#instancesigs)) · `TupleSections` — частичное применение конструктора кортежа: (,5) ([→](Extensions.ipynb#tuplesections))


## 📌 Содержание

| # | Раздел | Ключевая идея |
|---|--------|---------------|
| 1 | Категория: основание | id и (.) для любых морфизмов |
| 2 | Arrow: подъём и параллелизм | arr, first, (***), (&&&) |
| 3 | ArrowChoice: ветвление | left, right, (+++), (|||) |
| 4 | ArrowLoop: обратная связь | loop — traced monoidal |
| 5 | ArrowApply: стрелки первого класса | app — эквивалент монаде |
| 6 | Kleisli: Монада => Стрелка | Monad m => Arrow (Kleisli m) |
| 7 | Практические примеры | парсер, валидация, конвейер |
| 8 | Категорный взгляд | моноидальные категории, traced |

---

## 1. Категория: основание

### Определение

```haskell
class Category arr where
  id  :: arr a a
  (.) :: arr b c -> arr a b -> arr a c
```

Законы (те же, что для функций):
- **левое тождество:** `id . f = f`
- **правое тождество:** `f . id = f`
- **ассоциативность:** `(f . g) . h = f . (g . h)`

### Инстансы

| Тип | Что такое морфизм |
|-----|------------------|
| `(->)` | обычные функции |
| `Kleisli m` | монадические функции `a -> m b` |
| `Star f` | `a -> f b` (из профункторов) |

### Категорный взгляд

`Category` — это буквально математическая категория в Haskell:
объекты = типы, морфизмы = стрелки `arr a b`, композиция = `(.)`.

In [11]:
-- Category instance for (->) is built-in
-- Verify the laws

f :: Int -> Int
f x = x + 1

g :: Int -> Int
g x = x * 2

test1 :: Bool
test1 = (id . f) 5 == f 5

test2 :: Bool
test2 = (f . id) 5 == f 5

h :: Int -> String
h x = show x

test3 :: Bool
test3 = ((h . f) . g) 3 == (h . (f . g)) 3

print (test1, test2, test3)  -- (True, True, True)

Line 11: Redundant id
Found:
id . f
Why not:
fLine 14: Redundant id
Found:
(f . id) 5
Why not:
f 5Line 14: Redundant id
Found:
f . id
Why not:
fLine 17: Eta reduce
Found:
h x = show x
Why not:
h = show

(True,True,True)

---

## 2. Arrow: подъём и параллелизм

### Определение

```haskell
class Category arr => Arrow arr where
  arr   :: (a -> b) -> arr a b           -- подъём чистой функции
  first :: arr a b  -> arr (a,c) (b,c)   -- применить к левой части пары
```

Производные комбинаторы:

```haskell
(***)  :: arr a b -> arr c d -> arr (a,c) (b,d)  -- параллельное применение
(&&&)  :: arr a b -> arr a c -> arr a (b,c)       -- разветвление (fanout)
(>>>)  :: Category arr => arr a b -> arr b c -> arr a c
```

### Категорный взгляд

`(***)` — тензорное произведение моноидальной категории: `f ⊗ g`.
Пара `(Arrow, (***))` образует **строгую моноидальную категорию** с единицей `()`.

![Комбинаторы потока данных: параллельный, fanout, ветвление, цикл](../diagrams/arrows/arr_dataflow.svg)

In [12]:
double :: Int -> Int
double = (*2)

addOne :: Int -> Int
addOne = (+1)

-- (>>>) sequential composition
pipeline :: Int -> Int
pipeline = arr double >>> arr addOne

-- (***) parallel: double fst, addOne snd
bothOps :: (Int, Int) -> (Int, Int)
bothOps = arr double *** arr addOne

-- (&&&) fan-out: apply both to same input
forkOps :: Int -> (Int, Int)
forkOps = arr double &&& arr addOne

print (pipeline 5)        -- 11
print (bothOps (3, 7))    -- (6, 8)
print (forkOps 10)        -- (20, 11)

11

(6,8)

(20,11)

---

## 3. ArrowChoice: ветвление через Either

### Определение

```haskell
class Arrow arr => ArrowChoice arr where
  left  :: arr a b -> arr (Either a c) (Either b c)
  right :: arr a b -> arr (Either c a) (Either c b)
```

Производные комбинаторы:

```haskell
(+++) :: arr a b -> arr c d -> arr (Either a c) (Either b d)  -- выбор
(|||) :: arr a c -> arr b c -> arr (Either a b) c             -- слияние (merge)
```

### Категорный взгляд

`ArrowChoice` добавляет **копроизведение** в моноидальную категорию:
- `(+++)` — тензор на `Either` (бикартезианская структура)
- `(|||)` — коэкуализатор: «объединить обе ветки в одну»

In [13]:
routeEvenOdd :: Int -> Either Int Int
routeEvenOdd n = if even n then Left n else Right n

processEven :: Int -> String
processEven n = "even: " ++ show (n `div` 2)

processOdd :: Int -> String
processOdd n  = "odd:  " ++ show (n * 3 + 1)

-- (+++) applies different arrows to each side
processChoice :: Either Int Int -> Either String String
processChoice = arr processEven +++ arr processOdd

-- (|||) merges both branches
processAny :: Either Int Int -> String
processAny = arr processEven ||| arr processOdd

classifyAndProcess :: Int -> String
classifyAndProcess = arr routeEvenOdd >>> processAny

mapM_ (putStrLn . classifyAndProcess) [1..6]

odd:  4
even: 1
odd:  10
even: 2
odd:  16
even: 3

---

## 4. ArrowLoop: обратная связь (traced monoidal)

### Определение

```haskell
class Arrow arr => ArrowLoop arr where
  loop :: arr (a, c) (b, c) -> arr a b
```

`c` — внутреннее состояние, передаваемое обратно с выхода на вход.
Требование: `c` должно вычисляться лениво.

### Категорный взгляд

`ArrowLoop` соответствует **traced monoidal category**.
Операция `trace :: (A ⊗ C → B ⊗ C) → (A → B)` «закольцовывает» кабель C.
Формальное основание для циклов, счётчиков, рекурсивных схем.

In [14]:
import Data.List (unfoldr)

-- Running sum
runningSum :: [Int] -> [Int]
runningSum = scanl1 (+)

-- Fibonacci via unfoldr (loop pattern)
fibList :: [Integer]
fibList = unfoldr (\(a,b) -> Just (a, (b, a+b))) (0, 1)

-- loop on (->) instance
counterFrom :: Int -> [Int]
counterFrom n = loop go n
  where
    go :: (Int, [Int]) -> ([Int], [Int])
    go (start, _) = let xs = [start..start+4] in (xs, xs)

print (runningSum [1,2,3,4,5])   -- [1,3,6,10,15]
print (take 10 fibList)           -- [0,1,1,2,3,5,8,13,21,34]
print (counterFrom 3)             -- [3,4,5,6,7]

Line 13: Eta reduce
Found:
counterFrom n = loop go n
Why not:
counterFrom = loop go

[1,3,6,10,15]

[0,1,1,2,3,5,8,13,21,34]

[3,4,5,6,7]

---

## 5. ArrowApply: стрелки первого класса

### Определение

```haskell
class Arrow arr => ArrowApply arr where
  app :: arr (arr a b, a) b
```

`app` принимает стрелку как данные и применяет её — стрелки становятся значениями первого класса.

### Ключевая теорема: `ArrowApply ⟺ Monad`

- Каждая `Monad` даёт `ArrowApply` через `Kleisli`
- Каждый `ArrowApply` эквивалентен монаде

### Почему это важно

Обычный `Arrow` строже монады: стрелки не могут динамически выбирать
следующую стрелку на основе промежуточных значений — `arr` видит только типы.
`ArrowApply` снимает это ограничение, но теряет гарантии статического анализа.

In [15]:
-- Demonstrating Monad <=> ArrowApply equivalence

monadStyle :: Maybe Int
monadStyle = do
  x <- Just 5
  y <- Just (x * 2)
  return (x + y)

-- Kleisli arrow style
arrowStyle :: Maybe Int
arrowStyle = runKleisli pipeline3 5
  where
    pipeline3 =
      Kleisli (\x -> Just (x, x * 2)) >>>
      arr (\(a,b) -> a + b) >>>
      Kleisli Just

print monadStyle  -- Just 15
print arrowStyle  -- Just 15

Line 15: Use uncurry
Found:
\ (a, b) -> a + b
Why not:
uncurry (+)

Just 15

Just 15

---

## 6. Kleisli: каждая Монада даёт Стрелку

### Конструкция

```haskell
newtype Kleisli m a b = Kleisli { runKleisli :: a -> m b }

instance Monad m => Category (Kleisli m) where
  id                    = Kleisli return
  Kleisli f . Kleisli g = Kleisli (g >=> f)

instance Monad m => Arrow (Kleisli m) where
  arr f             = Kleisli (return . f)
  first (Kleisli f) = Kleisli (\(a,c) -> f a >>= \b -> return (b,c))
```

### Таблица соответствий

| Операция монады | Операция стрелки |
|----------------|-----------------|
| `return` | `arr id` |
| `>>=` | `>>>` (по существу) |
| `>=>` | `.` (композиция Kleisli) |
| `liftM` | `arr` |
| `join` | `app` (для ArrowApply) |

### Категорный взгляд

Категория Клейсли `Kl(m)`: объекты = типы Haskell,
морфизмы `a -> b` в `Kl(m)` = `a -> m b` в Hask.
`unit` и `join` монады становятся `id` и `(.)` в категории Клейсли.

![Конструкция Клейсли: Монада порождает Стрелку](../diagrams/arrows/arr_kleisli.svg)

In [16]:
-- Kleisli arrows for Maybe monad
safeDiv :: Int -> Kleisli Maybe Int Int
safeDiv d = Kleisli (\n -> if d == 0 then Nothing else Just (n `div` d))

safeSqrt :: Kleisli Maybe Int Double
safeSqrt = Kleisli (\n -> if n < 0 then Nothing else Just (sqrt (fromIntegral n)))

pipeline2 :: Kleisli Maybe Int Double
pipeline2 = safeDiv 2 >>> safeSqrt

test :: [Maybe Double]
test = map (runKleisli pipeline2) [100, 0, -4, 49]

-- Kleisli for [] (nondeterminism)
expand :: Kleisli [] Int Int
expand = Kleisli (\n -> [n-1, n, n+1])

expand2 :: Kleisli [] Int Int
expand2 = expand >>> expand

listResult :: [Int]
listResult = runKleisli expand2 0

print test        -- [Just 7.07..., Nothing, Nothing, Just 3.5]
print listResult  -- [-2,-1,0,-1,0,1,0,1,2]

[Just 7.0710678118654755,Just 0.0,Nothing,Just 4.898979485566356]

[-2,-1,0,-1,0,1,0,1,2]

---

## 7. Практические примеры: конвейер, валидация

### Классический стиль vs стрелочный

```haskell
-- Классический
process s = parseNum s >>= validatePositive >>= (Right . transform)

-- Стрелочный (статически инспектируемый)
processA = runKleisli $
  Kleisli parseNum >>>
  Kleisli validatePositive >>>
  arr transform
```

Стрелочный конвейер можно обходить, визуализировать, оптимизировать — не запуская.
Монадическая версия раскрывает структуру только во время выполнения.

In [17]:
parseNum :: String -> Either String Int
parseNum s = case reads s of
  [(n,"")] -> Right n
  _        -> Left ("parse error: " ++ s)

validatePositive :: Int -> Either String Int
validatePositive n
  | n > 0     = Right n
  | otherwise = Left ("must be positive, got: " ++ show n)

doubleIt :: Int -> Int
doubleIt = (*2)

processKleisli :: Kleisli (Either String) String Int
processKleisli =
  Kleisli parseNum >>>
  Kleisli validatePositive >>>
  arr doubleIt

mapM_ (print . runKleisli processKleisli) ["42", "abc", "-5", "7"]
-- Right 84 / Left "parse error: abc" / Left "must be positive..." / Right 14

Right 84
Left "parse error: abc"
Left "must be positive, got: -5"
Right 14

---

## 8. Категорный взгляд

### Иерархия стрелок

```
           Category
               |
            Arrow              моноидальная категория
           /   |    \
  ArrowChoice  |   ArrowLoop   бикартезианная + traced
              \|/
           ArrowApply           эквивалент монаде
```

### Arrow как моноидальная категория

`Arrow` с `(***)` образует **строгую моноидальную категорию**:
- Тензорное произведение: `(***)` на морфизмах, `(,)` на объектах
- Единица: `()`
- Закон тензора: `first (f >>> g) = first f >>> first g`

### ArrowLoop = Traced monoidal category

`ArrowLoop` добавляет операцию trace:
```
trace_C (f : A ⊗ C → B ⊗ C) : A → B
```
Позволяет строить циклические схемы, счётчики, рекурсивные цепи обратной связи.

### Связь с профункторами

`Arrow arr` — это `Strong` профунктор с дополнительной структурой:
- `arr f` поднимает чистые морфизмы (аналог `rmap`)
- `first` = применение к левой части пары (Strong)
- `ArrowApply` добавляет замкнутость (аналог профунктора `Closed`)

![Иерархия классов Arrow](../diagrams/arrows/arr_hierarchy.svg)

In [18]:
-- Verify arrow laws

-- law: arr (f . g) = arr f . arr g
law1_left :: Int -> String
law1_left = arr (show . (+1))

law1_right :: Int -> String
law1_right = arr (+1) >>> arr show

test_law1 :: Bool
test_law1 = map law1_left [1..5] == map law1_right [1..5]

-- law: first (arr f) = arr (bimap f id)
law2_left :: (Int, String) -> (Int, String)
law2_left = first (arr (+1))

law2_right :: (Int, String) -> (Int, String)
law2_right = arr (\(a,b) -> (a+1, b))

test_law2 :: Bool
test_law2 = map law2_left [(1,"a"),(2,"b")] == map law2_right [(1,"a"),(2,"b")]

print (test_law1, test_law2)  -- (True, True)

(True,True)

---

## Итоги

### Иерархия стрелок

| Класс | Ключевые операции | Категорный смысл |
|-------|------------------|-----------------|
| `Category` | `id`, `(.)` | Категория |
| `Arrow` | `arr`, `first`, `(***)`, `(&&&)` | Моноидальная категория |
| `ArrowChoice` | `left`, `(+++)`, `(|||)` | Бикартезианная моноидальная |
| `ArrowLoop` | `loop` | Traced monoidal category |
| `ArrowApply` | `app` | Эквивалент монаде |

### Когда использовать стрелки

- **Стрелки вместо монад**: когда структура вычисления должна быть статически анализируемой
- **Монады вместо стрелок**: когда нужно динамическое ветвление, зависящее от данных
- **Kleisli**: универсальный мост — каждая монада автоматически даёт стрелку

### Связи

- [Монады](Monads.ipynb) — каждая монада даёт стрелку `Kleisli`
- [Профункторы](Profunctors.ipynb) — `Arrow` — это `Strong` профунктор с дополнительной структурой
- [Сопряжения](Adjunctions.ipynb) — категория Клейсли и алгебра сопряжений
- [Иерархия функторов](FunctorHierarchy.ipynb) — стрелки как альтернативная система эффектов

---

**← Предыдущий:** [Оптики](Optics.ipynb)  |  **Следующий →** [Лемма Ёнеды](YonedaLemma.ipynb)
